# Notebook B v12.1 — Nemotron SFT with Assistant-Only Loss — CUTLASS fix

This is the patched version after the Kaggle run failed with `ModuleNotFoundError: No module named 'cutlass'`.

Fix: search both Kaggle utility paths:
- `/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script`
- `/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script`

Default experiment repeats the best 0.54 controlled bit-heavy mix, but trains with assistant-only loss.

In [ ]:
import os, re, gc, json, math, time, random, shutil, zipfile, stat, site, sys
from pathlib import Path
from collections import Counter, defaultdict

# -----------------------------
# Editable experiment config
# -----------------------------
TRACE_JSONL_NAME = "train_traces_v2_bit.jsonl"
OUTPUT_TAG = "adapter_sft_v2_bit_bal128_asstloss"

SAMPLE_PLAN = {
    "bit_manipulation": 65,
    "gravity": 21,
    "unit_conversion": 21,
    "numeral": 21,
}

RANDOM_SEED = 42
MAX_SEQ_LEN = 384
LR = 2e-4
NUM_EPOCHS = 1
GRAD_ACCUM_STEPS = 1
LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
USE_ASSISTANT_ONLY_LOSS = True

WORKING_DIR = Path('/kaggle/working')
OUTPUT_DIR = WORKING_DIR / OUTPUT_TAG
SMOKE_ZIP_PATH = WORKING_DIR / f"{OUTPUT_TAG}.zip"

print('OUTPUT_TAG:', OUTPUT_TAG)
print('TRACE_JSONL_NAME:', TRACE_JSONL_NAME)
print('SAMPLE_PLAN:', SAMPLE_PLAN)
print('MAX_SEQ_LEN:', MAX_SEQ_LEN)
print('USE_ASSISTANT_ONLY_LOSS:', USE_ASSISTANT_ONLY_LOSS)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('SMOKE_ZIP_PATH:', SMOKE_ZIP_PATH)

In [ ]:
# -----------------------------
# Kaggle / Nemotron runtime setup — robust CUTLASS path fix
# -----------------------------
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

UTILITY_ROOTS = [
    Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script'),
    Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script'),
    Path('/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script'),
    Path('/tmp'),
]

print('Utility root discovery:')
for root in UTILITY_ROOTS:
    print(' -', root, 'exists=', root.exists())
    if root.exists():
        sys.path.insert(0, str(root))
        site.addsitedir(str(root))
        cutlass_path = root / 'nvidia_cutlass_dsl' / 'python_packages'
        print('   cutlass_path:', cutlass_path, 'exists=', cutlass_path.exists())
        if cutlass_path.exists():
            sys.path.insert(0, str(cutlass_path))
            site.addsitedir(str(cutlass_path))

# Verify cutlass before model import.
try:
    import cutlass
    print('CUTLASS import PASS:', getattr(cutlass, '__file__', 'unknown'))
except Exception as e:
    print('CUTLASS import FAILED after path setup:', repr(e))
    print('First 20 sys.path entries:')
    for p in sys.path[:20]:
        print('  ', p)
    raise

# Make ptxas binaries executable from whichever utility root exists.
for root in UTILITY_ROOTS:
    for src in [
        root / 'triton/backends/nvidia/bin/ptxas',
        root / 'triton/backends/nvidia/bin/ptxas-blackwell',
    ]:
        if src.exists():
            dst = Path('/tmp') / src.name
            shutil.copy2(src, dst)
            os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
            print('Prepared executable:', dst)
            if dst.name == 'ptxas':
                os.environ['TRITON_PTXAS_PATH'] = str(dst)
            if dst.name == 'ptxas-blackwell':
                os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = str(dst)

try:
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Patched Triton ptxas version check.')
except Exception as e:
    print('Triton patch skipped:', repr(e))

def patch_nemotron_fast_path():
    patched = []
    for name, mod in list(sys.modules.items()):
        if 'modeling_nemotron_h' in name and hasattr(mod, 'is_fast_path_available'):
            try:
                mod.is_fast_path_available = False
                patched.append(name)
            except Exception:
                pass
    if patched:
        print('Patched fast path modules:', patched)

patch_nemotron_fast_path()

In [ ]:
# -----------------------------
# Locate and load Notebook A traces
# -----------------------------
def find_trace_jsonl(name: str) -> Path:
    roots = [Path('/kaggle/input/notebooks'), Path('/kaggle/input'), WORKING_DIR]
    hits = []
    for root in roots:
        if root.exists():
            hits.extend(root.glob(f'**/{name}'))
    hits = sorted(set(hits), key=lambda p: str(p))
    print('Trace candidates:')
    for h in hits[:20]:
        print(' -', h)
    if not hits:
        raise FileNotFoundError(f'Could not find {name}')
    return hits[0]

TRACE_PATH = find_trace_jsonl(TRACE_JSONL_NAME)
print('Using TRACE_PATH:', TRACE_PATH)
records = []
with open(TRACE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print('Loaded trace records:', len(records))
print('Category counts:', Counter(r.get('category', 'UNKNOWN') for r in records))
print('Example keys:', sorted(records[0].keys()))
print('Example category:', records[0].get('category'))

In [ ]:
# -----------------------------
# Deterministic sampling
# -----------------------------
random.seed(RANDOM_SEED)
by_cat = defaultdict(list)
for r in records:
    by_cat[r.get('category', 'UNKNOWN')].append(r)

selected = []
for cat, n in SAMPLE_PLAN.items():
    pool = list(by_cat.get(cat, []))
    if len(pool) < n:
        raise ValueError(f'Not enough records for {cat}: requested {n}, available {len(pool)}')
    rng = random.Random(RANDOM_SEED + abs(hash(cat)) % 100000)
    rng.shuffle(pool)
    selected.extend(pool[:n])
random.Random(RANDOM_SEED).shuffle(selected)
print('Selected records:', len(selected))
print('Selected category counts:', Counter(r.get('category', 'UNKNOWN') for r in selected))
assert len(selected) == sum(SAMPLE_PLAN.values())
print('Sampling validation passed.')

In [ ]:
# -----------------------------
# Load model/tokenizer and apply LoRA
# -----------------------------
import torch
import kagglehub
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
print('MODEL_PATH:', MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, trust_remote_code=True, dtype=torch.bfloat16, device_map={'': 0}
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16, device_map={'': 0}
    )

patch_nemotron_fast_path()
model.config.use_cache = False
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=r'.*\\.(in_proj|out_proj|up_proj|down_proj)$',
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Model + LoRA ready.')

In [ ]:
# -----------------------------
# Build chat text and assistant-only labels
# -----------------------------
def boxed(answer):
    s = str(answer).strip()
    return s if '\\boxed{' in s else f'\\boxed{{{s}}}'

def record_to_messages(rec):
    if isinstance(rec.get('messages'), list) and len(rec['messages']) >= 2:
        return rec['messages']
    prompt = rec.get('prompt') or rec.get('user') or rec.get('question') or rec.get('input')
    assistant = rec.get('assistant') or rec.get('response') or rec.get('completion') or rec.get('solution')
    answer = rec.get('answer')
    if assistant is None:
        assistant = boxed(answer)
    elif answer is not None and '\\boxed{' not in str(assistant):
        assistant = str(assistant).rstrip() + '\n\nFinal answer: ' + boxed(answer)
    if prompt is None:
        return None
    user_msg = str(prompt).rstrip() + '\nPlease put your final answer inside `\\boxed{}`.'
    return [{'role': 'user', 'content': user_msg}, {'role': 'assistant', 'content': str(assistant).strip()}]

def encode_record(rec):
    messages = record_to_messages(rec)
    if messages is None:
        text = rec.get('text') or rec.get('formatted_text')
        if not text:
            raise ValueError(f'Cannot build text for record keys={sorted(rec.keys())}')
        enc = tokenizer(str(text), truncation=True, max_length=MAX_SEQ_LEN, return_tensors='pt')
        labels = enc['input_ids'].clone()
        return enc['input_ids'][0], enc['attention_mask'][0], labels[0], 'full_text_fallback'

    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    full = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LEN, return_tensors='pt')
    input_ids = full['input_ids'][0]
    attention_mask = full['attention_mask'][0]
    labels = input_ids.clone()
    if USE_ASSISTANT_ONLY_LOSS:
        user_only = [m for m in messages if m.get('role') != 'assistant']
        prefix_text = tokenizer.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
        prefix_ids = tokenizer(prefix_text, add_special_tokens=False, return_tensors='pt')['input_ids'][0]
        prefix_len = min(len(prefix_ids), len(labels))
        labels[:prefix_len] = -100
        if (labels != -100).sum().item() == 0:
            return None
    return input_ids, attention_mask, labels, 'assistant_only' if USE_ASSISTANT_ONLY_LOSS else 'full_text'

encoded = []
skipped = 0
modes = Counter()
for rec in selected:
    item = encode_record(rec)
    if item is None:
        skipped += 1
        continue
    encoded.append((rec, *item))
    modes[item[-1]] += 1
print('Encoded examples:', len(encoded), 'skipped:', skipped)
print('Label modes:', modes)
assert encoded
rec, input_ids, attention_mask, labels, mode = encoded[0]
print('First encoded length:', len(input_ids), 'mode:', mode, 'category:', rec.get('category'))
print('Train-label tokens:', int((labels != -100).sum().item()))

In [ ]:
# -----------------------------
# Manual SFT loop
# -----------------------------
model.train()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
losses = []
step = 0
start = time.time()
for epoch in range(NUM_EPOCHS):
    random.Random(RANDOM_SEED + epoch).shuffle(encoded)
    for i, (rec, input_ids, attention_mask, labels, mode) in enumerate(encoded, start=1):
        input_ids = input_ids.unsqueeze(0).to(model.device)
        attention_mask = attention_mask.unsqueeze(0).to(model.device)
        labels = labels.unsqueeze(0).to(model.device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()
        if i % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
        losses.append(float(loss.detach().cpu()) * GRAD_ACCUM_STEPS)
        if i == 1 or i % 10 == 0 or i == len(encoded):
            elapsed = time.time() - start
            print(f'epoch={epoch+1} item={i}/{len(encoded)} step={step} loss={losses[-1]:.4f} elapsed={elapsed:.1f}s cat={rec.get("category")}')
        del outputs, loss, input_ids, attention_mask, labels
        if i % 25 == 0:
            gc.collect()
            torch.cuda.empty_cache()
print('Training complete.')
print('Mean loss:', sum(losses) / len(losses))
print('Last 10 mean loss:', sum(losses[-10:]) / min(10, len(losses)))

In [ ]:
# -----------------------------
# Save adapter and create debug zip for Notebook C
# -----------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tok_dbg = OUTPUT_DIR / 'tokenizer_debug'
tok_dbg.mkdir(exist_ok=True)
try:
    tokenizer.save_pretrained(tok_dbg)
except Exception as e:
    print('Tokenizer debug save skipped:', repr(e))
print('Saved adapter files:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print(' -', p.relative_to(OUTPUT_DIR), f'{p.stat().st_size/1024/1024:.2f} MB')
if SMOKE_ZIP_PATH.exists():
    SMOKE_ZIP_PATH.unlink()
with zipfile.ZipFile(SMOKE_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['adapter_config.json', 'adapter_model.safetensors', 'adapter_model.bin']:
        p = OUTPUT_DIR / fname
        if p.exists():
            zf.write(p, arcname=fname)
with zipfile.ZipFile(SMOKE_ZIP_PATH, 'r') as zf:
    names = zf.namelist()
print('Created adapter debug zip:', SMOKE_ZIP_PATH, f'{SMOKE_ZIP_PATH.stat().st_size/1024/1024:.2f} MB')
print('Zip contents:', names)
assert 'adapter_config.json' in names
assert ('adapter_model.safetensors' in names) or ('adapter_model.bin' in names)
print('Notebook B complete. Use this exact tag in Notebook C:', OUTPUT_TAG)